# Lab 1 — คลังความจำ เขียนเอง (EP12)

คลิปทฤษฎีเปิดด้วย research assistant ที่วันอังคารจำเรื่องวันจันทร์ไม่ได้เลย
Lab นี้เราจะเห็นปัญหานั้นจริง แล้วเขียน **คลังความจำ** ตาม Listing 17.1 ของตำรา (§17.4)

**สิ่งที่จะได้ทำ**
1. คุยวันจันทร์ แล้วถามวันอังคาร **โดยไม่มีความจำ**
2. `MemoryEntry` — ความจำหนึ่งชิ้น: ข้อความ, embedding, เวลา, ความสำคัญ
3. `VectorMemoryStore` — **write** (กันข้อมูลซ้ำ) · **retrieve** (ความเหมือน + ความใหม่) · **evict** (ลบเมื่อเต็ม)
4. วันอังคาร **พร้อมความจำ** — และเหตุที่ครั้งแรกยังพลาด

เวลาของความจำเป็นเวลาจำลอง ("เมื่อวาน", "สองปีก่อน") ส่วนคำตอบของ model ทุกคำตอบมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.1, §17.4, §17.5.2, Listing 17.1 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **5 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep12-lab1-memory-store.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | สมัครที่ openrouter.ai → Keys → Create Key แล้วเติมเครดิต (lab นี้ใช้ไม่ถึง 0.1 เซ็นต์ดอลลาร์) |
| Packages | `openai`, `python-dotenv`, `numpy` (ติดตั้งใน cell ถัดไป) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

⚠️ อย่าใส่ key ลงในโค้ดโดยตรง และอย่าแชร์ไฟล์ `.env`

In [ ]:
%pip install -q "openai>=1.50" python-dotenv numpy

## 1. Model สองตัว: ตัวตอบ และตัวทำ embedding

**ตัวตอบ `google/gemma-4-26b-a4b-it`** — model เดียวกับทุก EP ล็อก provider เป็น Makora
ราคา 0.09 ดอลลาร์ต่อล้าน token ขาเข้า และ 0.28 ต่อล้าน token ขาออก
· รุ่น `:free` ใช้ไม่ได้ (ทดสอบ 2026-11-22: 3 ใน 3 ครั้งได้ 429 "rate-limited")

**ตัวทำ embedding `qwen/qwen3-embedding-8b`** — ตัวเดียวกับ EP10–EP11 ล็อก provider เป็น Nebius
— ความจำแต่ละชิ้นกลายเป็นเวกเตอร์ เพื่อค้นด้วยความหมาย

ค่าใช้จ่ายจริงของทุกคำขอรวมไว้ใน `COST`

In [ ]:
import os
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import numpy as np
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

In [ ]:
KEY = load_key()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY, max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def chat(messages):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=messages)
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=[text], extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    v = np.array(r.data[0].embedding)
    return v / np.linalg.norm(v)                         # length 1: dot product = cosine

## 2. วันจันทร์ และวันอังคาร — ไม่มีความจำ

LLM ไม่มีสถานะ (§17.1): ทุกครั้งที่เรียก มันเห็นแค่ข้อความที่เราส่งไปในครั้งนั้น
วันอังคารคือ **session ใหม่** — รายการข้อความเริ่มจากว่าง

In [ ]:
monday = [{"role": "user", "content": "I'm writing a report on transformers for time series. "
                                      "Please keep your answers short."}]
print(chat(monday))

In [ ]:
tuesday = [{"role": "user", "content": "Any new papers for my report?"}]
print(chat(tuesday)[:600])

## 3. ความจำหนึ่งชิ้น: `MemoryEntry`

ตาม Listing 17.1: ข้อความ, embedding, **เวลา** ที่เขียน, **ความสำคัญ** (0–1), จำนวนครั้งที่ถูกเรียก และแหล่งที่มา

In [ ]:
@dataclass
class MemoryEntry:
    content: str
    embedding: np.ndarray
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5
    access_count: int = 0
    last_accessed: datetime | None = None
    source: str = "conversation"

## 4. คลังความจำ: `VectorMemoryStore`

| method | ทำอะไร |
|---|---|
| `similarities(q)` | cosine ของเวกเตอร์ `q` กับทุกความจำ |
| `recency(entry, now)` | ความใหม่ `exp(−decay × ชั่วโมง)` — 1 คือเพิ่งเขียน |
| `write(...)` | ข้ามถ้าซ้ำ (cosine > 0.95) · ถ้าเต็ม → `evict()` · แล้วเก็บ |
| `evict()` | ลบชิ้นที่ `importance × 0.7 + recency × 0.3` ต่ำสุด |
| `retrieve(q, k)` | คะแนน `0.7 × sim + 0.3 × recency` → k ชิ้นแรก |

`decay_rate` ของตำราคือ 0.01 ต่อชั่วโมง (ครึ่งหนึ่งใน ~3 วัน) — ความจำของเรากินเวลาเป็นปี จึงใช้ 0.001 (ครึ่งหนึ่งใน ~29 วัน)

In [ ]:
class VectorMemoryStore:
    def __init__(self, embed_fn, max_entries=10_000, decay_rate=0.001, recency_weight=0.3):
        self.embed_fn = embed_fn            # text -> unit vector
        self.max_entries = max_entries
        self.decay_rate = decay_rate        # per hour (the book uses 0.01)
        self.recency_weight = recency_weight
        self.entries = []

    # ✏️ TODO 1 (▶ วิดีโอ 3:57): เรียงเวกเตอร์ของทุกความจำเป็นเมทริกซ์ แล้วคูณกับ q ได้ cosine ทุกชิ้นพร้อมกัน

    def similarities(self, q):
        return ___

    # ✏️ TODO 2 (▶ วิดีโอ 4:12): คิดอายุเป็นชั่วโมง แล้วคืน exp(−decay_rate × ชั่วโมง)

    def recency(self, entry, now):
        hours = ___
        return ___

    def write(self, content, importance=0.5, when=None, check_duplicates=True):
        emb = self.embed_fn(content)
        closest = self.similarities(emb).max() if self.entries else 0.0
        if check_duplicates and closest > 0.95:
            print(f"skip   (closest {closest:.2f}): {content}")
            return None
        if len(self.entries) >= self.max_entries:
            self.evict()
        entry = MemoryEntry(content, emb, when or datetime.now(), importance)
        self.entries.append(entry)
        print(f"stored (closest {closest:.2f}): {content}")
        return entry

    def evict(self):
        now, w = datetime.now(), self.recency_weight
        scores = [e.importance * (1 - w) + self.recency(e, now) * w for e in self.entries]
        # ✏️ TODO 3 (▶ วิดีโอ 5:14): ลบความจำที่คะแนนต่ำสุดออกจาก self.entries
        gone = ___
        print(f"evict  (score {min(scores):.2f}): {gone.content}")

    # ✏️ TODO 4 (▶ วิดีโอ 5:19): คะแนนรวม = (1 − w) × ความเหมือน + w × ความใหม่ (หรือความเหมือนอย่างเดียว ถ้าปิด recency_boost)

    def retrieve(self, query, k=5, recency_boost=True):
        sims = self.similarities(self.embed_fn(query))
        now, w = datetime.now(), self.recency_weight
        scores = [___ if recency_boost else s
                  for e, s in zip(self.entries, sims)]
        top = list(np.argsort(scores)[::-1][:k])
        for i in top:
            self.entries[i].access_count += 1
            self.entries[i].last_accessed = now
        return [(scores[i], sims[i], self.entries[i]) for i in top]

In [ ]:
def show(hits):
    for score, sim, e in hits:
        age = (datetime.now() - e.timestamp).days
        print(f"{score:.2f}  sim {sim:.2f}  {age:>3} days  {e.content}")

## 5. เขียนความจำของวันจันทร์

คลังขนาดเล็ก (`max_entries=6`) เพื่อให้เห็นการลบเมื่อเต็ม · เวลาเป็นเวลาจำลอง: **เมื่อวาน**
ความสำคัญเราตั้งเองไปก่อน — Lab 2 ให้ model ให้คะแนนเอง

In [ ]:
mem = VectorMemoryStore(embed, max_entries=6)
yesterday = datetime.now() - timedelta(days=1)               # simulated: Monday
mem.write("The user is writing a report on transformers for time series.", importance=0.9, when=yesterday)
mem.write("The user prefers short answers.", importance=0.8, when=yesterday)
mem.write("The user said thanks.", importance=0.1, when=yesterday)
mem.write("The user prefers short answers.", importance=0.8, when=yesterday)
mem.write("The user wants answers kept brief.", importance=0.8, when=yesterday)
print(len(mem.entries), "memories")

## 6. ค้นด้วยความเหมือน + ความใหม่

สองความจำเรื่อง editor: **vim** (สองปีก่อน) และ **VS Code** (สัปดาห์ก่อน) — ข้อเท็จจริงเก่าที่ถูกแทนที่แล้ว

In [ ]:
mem.write("The user's code editor is vim.", importance=0.6,
          when=datetime.now() - timedelta(days=730))
mem.write("The user switched their code editor to VS Code.", importance=0.6,
          when=datetime.now() - timedelta(days=7))
show(mem.retrieve("Which code editor do I use?", k=2, recency_boost=False))
print()
show(mem.retrieve("Which code editor do I use?", k=2))

## 7. คลังเต็ม: ลบชิ้นที่ทั้งไม่สำคัญและเก่า

ตอนนี้มี 6 ชิ้น = `max_entries` · ชิ้นที่ 7 ทำให้ต้องลบหนึ่งชิ้น

In [ ]:
mem.write("The user's report is due on Friday.", importance=0.8)
print(len(mem.entries), "memories")

## 8. วันอังคาร — พร้อมความจำ (session continuity, §17.5.2)

ก่อนตอบ: ค้นความจำที่เกี่ยวกับคำถาม → ใส่ใน system prompt → ถาม model

In [ ]:
def tuesday(question, k):
    hits = mem.retrieve(question, k=k)
    show(hits)
    # ✏️ TODO 5 (▶ วิดีโอ 8:14): ใส่ความจำที่ค้นได้ลงใน system prompt บรรทัดละชิ้น แล้วถาม model
    system = "What you remember about this user:\n" + ___
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": question}])

print(tuesday("Any new papers for my report?", k=3))

In [ ]:
print(tuesday("Any new papers for my report?", k=4))

In [ ]:
print({k: f"${v:.5f}" for k, v in COST.items()})

## สรุป

| ขั้น | สิ่งที่เห็น |
|---|---|
| วันอังคาร ไม่มีความจำ | ถามหัวข้อ + ตอบยาว — ลืมทั้งสองเรื่อง |
| write: ซ้ำคำต่อคำ | ข้าม (closest 1.00) |
| write: คนละคำ ความหมายเดียวกัน | เก็บ (closest 0.94 < 0.95) |
| ถาม editor, ความเหมือนอย่างเดียว | VS Code 0.77 · vim 0.74 — ข้อมูลขัดกัน |
| ถาม editor, + ความใหม่ | VS Code อันดับ 1 · vim หลุด (แต่ "short answers" ขึ้นมาแทน) |
| คลังเต็ม | ลบ "said thanks" (0.36) |
| วันอังคาร k=3 | หัวข้อหลุด — ความจำซ้ำกินสองช่อง → ยังถามหัวข้อ |
| วันอังคาร k=4 | ตอบเรื่อง transformer สำหรับ time series สั้นๆ |

**แบบฝึกหัด**
1. ตั้ง `decay_rate=0.01` (ค่าของตำรา) แล้วถามเรื่อง editor อีกครั้ง — อะไรขึ้นมาบ้าง และทำไม
2. ลดเกณฑ์กันซ้ำเป็น 0.90 — "wants answers kept brief" ถูกข้ามไหม แล้วความจำเรื่อง vim กับ VS Code ถูกข้ามด้วยหรือเปล่า
3. เขียน `get_stats()` ตามตำรา: จำนวนความจำ ความสำคัญเฉลี่ย และความจำที่เก่าที่สุด

**ถัดไป — Lab 2:** ให้ model ตัดสินเองว่าอะไรควรจำ ดึงเป็นข้อเท็จจริงสั้นๆ รวมความจำที่ซ้ำ จัดการความจำที่ขัดกัน และทบทวนเป็นบทเรียน